# Student Performance Analysis and Prediction Using Data Analytics and Machine Learning

**Student:** Hemant Sharma  
**Roll No.:** 24202855  
**Enrollment No.:** 23/133726  
**College:** Govt. P.G. College, Dholpur  
**University:** Maharaja Surajmal Brij University  

> Academic demonstration project. The included dataset is synthetic and does not represent real students. It is designed to demonstrate a reproducible data analytics workflow, not to make decisions about actual learners.

## 1. Objectives
- Inspect and validate the dataset.
- Summarize student learning indicators.
- Visualize patterns and relationships.
- Compare baseline regression models using a fixed train/test split.
- Discuss ethical limits and appropriate use of predictions.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
DATA_PATH = "student_performance_dataset.csv"
df = pd.read_csv(DATA_PATH)
df.head()

## 2. Data dictionary
`student_id` is a synthetic row identifier. `attendance_pct` and `assignment_completion_pct` are percentages. `study_hours_per_day`, `sleep_hours`, and `extracurricular_hours_per_week` are numeric quantities. `previous_score` and `final_score` are scores on a 0–100 scale. `internet_access` and `parental_support` are categorical fields.

In [ ]:
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nNumeric summary:")
df.describe().round(2)

## 3. Descriptive analysis

In [ ]:
numeric_cols = ["attendance_pct", "study_hours_per_day", "assignment_completion_pct", "previous_score", "sleep_hours", "extracurricular_hours_per_week", "final_score"]
summary = df[numeric_cols].agg(["mean", "median", "std", "min", "max"]).T.round(2)
summary

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(df["final_score"], bins=18, edgecolor="black")
plt.title("Distribution of Final Scores")
plt.xlabel("Final score (0–100)")
plt.ylabel("Number of synthetic records")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.scatter(df["study_hours_per_day"], df["final_score"], alpha=0.55)
plt.title("Study Time and Final Score")
plt.xlabel("Study hours per day")
plt.ylabel("Final score (0–100)")
plt.tight_layout()
plt.show()

## 4. Prepare the modelling data
The target variable is `final_score`. The synthetic identifier is excluded. Categorical variables are encoded using one-hot encoding. The test set is held out from model fitting; random state 42 makes the split repeatable. This split is an instructional benchmark, not independent external validation.

In [ ]:
target = "final_score"
X = df.drop(columns=["student_id", target])
y = df[target]
X = pd.get_dummies(X, columns=["internet_access", "parental_support"], drop_first=True, dtype=int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Model features:", list(X.columns))

## 5. Train and compare models
Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE) measure typical prediction error in score points; lower values are better. R² compares model performance to predicting the mean of the test target; higher values are generally better, though R² can be negative.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=250, max_depth=8, min_samples_leaf=3, random_state=RANDOM_STATE
    ),
}
rows = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred),
    })
metrics = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
metrics.round(3)

In [ ]:
best_name = metrics.loc[0, "Model"]
best_model = models[best_name]
best_pred = predictions[best_name]
print("Selected model by lowest test RMSE:", best_name)
print(metrics.iloc[0].to_dict())
plt.figure(figsize=(6.5, 5))
plt.scatter(y_test, best_pred, alpha=0.65)
plt.plot([0, 100], [0, 100], linestyle="--")
plt.xlabel("Actual score")
plt.ylabel("Predicted score")
plt.title(f"Actual vs Predicted Scores — {best_name}")
plt.tight_layout()
plt.show()

## 6. Feature importance
For a Random Forest model, impurity-based feature importance can help describe which input variables were useful in the fitted model. It is not proof of causation and may be biased by correlated features. For Linear Regression, absolute coefficient magnitude is shown as a rough comparison only; features are on different scales, so coefficient magnitudes should not be interpreted as a standardized ranking.

In [ ]:
if best_name == "Random Forest Regressor":
    importance = pd.Series(best_model.feature_importances_, index=X.columns)
    x_label = "Relative importance"
else:
    importance = pd.Series(np.abs(best_model.coef_), index=X.columns)
    x_label = "Absolute coefficient (features are not standardized)"
importance = importance.sort_values().tail(8)
importance

In [ ]:
importance.sort_values().plot(kind="barh", figsize=(8, 4.5))
plt.title(f"Top Predictive Features — {best_name}")
plt.xlabel(x_label)
plt.tight_layout()
plt.show()

## 7. Example prediction
This sample row is hypothetical and uses the same feature names and encoding as the training data. It must not be treated as advice or an assessment of any real student.

In [ ]:
sample = pd.DataFrame([{"attendance_pct": 85.0, "study_hours_per_day": 3.0,
    "assignment_completion_pct": 82.0, "previous_score": 70.0,
    "sleep_hours": 7.0, "internet_access": "Yes",
    "extracurricular_hours_per_week": 1.0, "parental_support": "Medium"}])
sample = pd.get_dummies(sample, columns=["internet_access", "parental_support"], drop_first=True, dtype=int)
sample = sample.reindex(columns=X.columns, fill_value=0)
print("Illustrative predicted score:", round(float(best_model.predict(sample)[0]), 1))

## 8. Findings and limitations
- The dataset is synthetic; relationships were constructed for teaching purposes.
- Results depend on the random seed, train/test split, selected variables and model settings.
- The test set is not a substitute for validation on real, representative data.
- Correlation and feature importance do not establish causation.
- A model should not be used for student ranking, admission, discipline or other high-impact decisions.
- A real deployment would require consent, data minimization, fairness checks, domain review, monitoring and a clear human-oversight process.

## 9. Reproducibility
Run the cells from top to bottom in the folder containing `student_performance_dataset.csv`. Install dependencies using `pip install -r requirements.txt`. The seed is fixed at 42.

## References
1. pandas documentation: https://pandas.pydata.org/docs/
2. scikit-learn documentation: https://scikit-learn.org/stable/
3. Matplotlib documentation: https://matplotlib.org/stable/
4. Jupyter documentation: https://docs.jupyter.org/
